# 🎙️ Voz local · XTTS v2
Clone grátis do ElevenLabs: texto em português → fala com a voz de um áudio de 5 a 10 s.

1. Menu **Ambiente de execução → Alterar o tipo de ambiente → GPU T4** → Salvar.
2. Menu **Ambiente de execução → Executar tudo**.
3. No fim aparece um link `https://….gradio.live`: abra, cole o texto, envie a voz e clique em **Gerar Voz Clonada**.

A primeira geração baixa o modelo (~1,8 GB, uns 2 min). Só clone a sua voz ou a de quem autorizou.

In [ ]:
#@title 1 · Conferir a GPU
!nvidia-smi -L || echo 'Sem GPU: troque o ambiente para GPU T4 (fica bem mais rápido)'

In [ ]:
#@title 2 · Instalar (uns 2 min)
!pip install -q coqui-tts gradio soundfile

In [ ]:
%%writefile motor.py
"""Motor de voz local: XTTS v2 clona uma voz a partir de 5 a 10 s de áudio de referência.

Uso direto:  python motor.py "texto em português" voz_modelo.wav saida.mp3
"""
import os
import re
import subprocess
import sys

import numpy as np
import soundfile as sf
import torch

os.environ.setdefault("COQUI_TOS_AGREED", "1")  # licença CPML da Coqui: uso pessoal, não comercial
from TTS.api import TTS

SR = 24000  # taxa de saída do XTTS v2
_xtts = None
_latentes = {}  # referência -> (gpt_cond_latent, speaker_embedding), calculado uma vez por voz


def xtts():
    """Carrega o modelo uma vez só, na GPU se houver."""
    global _xtts
    if _xtts is None:
        disp = "cuda" if torch.cuda.is_available() else "cpu"
        print("Carregando XTTS v2 em", disp, "(o primeiro uso baixa ~1,8 GB)")
        _xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(disp).synthesizer.tts_model
    return _xtts


def frases(texto):
    """Uma frase por vez: o XTTS erra em trechos longos e assim dá para pôr pausa de verdade.
    Só quebra em pontuação seguida de espaço, então "40.000" fica inteiro."""
    return [f.strip() for f in re.split(r"(?<=[.!?:;])\s+", texto.strip()) if f.strip()]


def gerar(texto, referencia, saida="saida.wav", pausa=0.45, velocidade=1.0):
    """Gera `texto` com a voz de `referencia` e salva em .wav ou .mp3. Devolve o caminho."""
    m = xtts()
    if referencia not in _latentes:
        _latentes[referencia] = m.get_conditioning_latents(audio_path=[referencia])
    gpt, spk = _latentes[referencia]
    partes = []
    for f in frases(texto):
        with torch.inference_mode():
            wav = m.inference(f, "pt", gpt, spk, speed=velocidade, temperature=0.7)["wav"]
        partes.append(np.asarray(wav, dtype=np.float32))
        # pausa menor depois de dois-pontos e ponto e vírgula
        partes.append(np.zeros(int((pausa * 0.55 if f[-1] in ":;" else pausa) * SR), np.float32))
    audio = np.concatenate(partes[:-1]) if partes else np.zeros(SR // 2, np.float32)
    audio = audio / max(1e-6, np.abs(audio).max()) * 0.9

    wav_path = saida if saida.endswith(".wav") else saida.rsplit(".", 1)[0] + ".wav"
    sf.write(wav_path, audio, SR)
    if saida.endswith(".mp3"):  # precisa do ffmpeg instalado
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav_path, "-b:a", "128k", saida], check=True)
        os.remove(wav_path)
    return saida


if __name__ == "__main__":
    if len(sys.argv) < 3:
        sys.exit(__doc__)
    print(gerar(sys.argv[1], sys.argv[2], sys.argv[3] if len(sys.argv) > 3 else "saida.wav"))


In [ ]:
%%writefile app.py
"""Interface web do motor de voz. Rode: python app.py  (no Colab: python app.py --share)"""
import sys

import gradio as gr

from motor import gerar


def acao(texto, referencia, pausa, velocidade):
    if not texto or not texto.strip():
        raise gr.Error("Escreva o texto.")
    if not referencia:
        raise gr.Error("Envie o áudio de referência (5 a 10 s).")
    return gerar(texto, referencia, "resultado.wav", pausa, velocidade)


with gr.Blocks(title="Voz local") as demo:
    gr.Markdown("# Voz local · XTTS v2\nClone só a sua voz ou a de quem autorizou.")
    texto = gr.Textbox(label="Texto em português", lines=6)
    referencia = gr.Audio(label="Voz de referência (5 a 10 s, fala limpa, sem música)", type="filepath")
    with gr.Row():
        pausa = gr.Slider(0.1, 1.5, value=0.45, step=0.05, label="Pausa entre frases (s)")
        velocidade = gr.Slider(0.7, 1.3, value=1.0, step=0.05, label="Velocidade")
    botao = gr.Button("Gerar Voz Clonada", variant="primary")
    resultado = gr.Audio(label="Resultado (o player tem botão de baixar)", type="filepath")
    botao.click(acao, [texto, referencia, pausa, velocidade], resultado)

if __name__ == "__main__":
    demo.queue().launch(share="--share" in sys.argv)


In [ ]:
#@title 3 · Abrir o app (clique no link gradio.live)
!python app.py --share